In [ ]:
import pymysql
import time
from getpass import getpass

def log(msg: str) -> None:
    now = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{now}] {msg}")

try:
    # ---------------------------------------------------------
    # CONNECT
    # ---------------------------------------------------------
    log("Connecting to MySQL...")
    conn = pymysql.connect(
        host="localhost",
        user="Oyeniyi_ETL",
        password=getpass("MySQL password: "),
        database="electricity_capstone",
        autocommit=False
    )
    cursor = conn.cursor()
    log("Connection established.")

    # ---------------------------------------------------------
    # METADATA
    # ---------------------------------------------------------
    batch_id = int(time.time())
    pipeline_id = "etl_v1"

    log("---------------------------------------------------------")
    log(f"Starting FACT LOAD | batch_id={batch_id} | pipeline_id={pipeline_id}")
    log("---------------------------------------------------------")

    # ---------------------------------------------------------
    # FACT_DEMAND
    # ---------------------------------------------------------
    log("Counting fact_demand rows to insert...")

    cursor.execute("""
        SELECT COUNT(*)
        FROM tmp_cleaned_demand td
        JOIN dim_time dt ON dt.datetime_utc = td.datetime_utc
        JOIN dim_country dc
            ON dc.country_name COLLATE utf8mb4_unicode_ci =
               td.country_code COLLATE utf8mb4_unicode_ci
        LEFT JOIN fact_demand fd
            ON fd.time_id = dt.time_id
           AND fd.country_id = dc.country_id
        WHERE fd.demand_id IS NULL;
    """)
    fact_demand_source = cursor.fetchone()[0]
    log(f"fact_demand rows to insert: {fact_demand_source}")

    t0 = time.time()
    log("Loading fact_demand...")

    cursor.execute("""
        INSERT INTO fact_demand (
            time_id,
            country_id,
            cov_ratio,
            value,
            value_scaled,
            batch_id,
            pipeline_id,
            load_ts
        )
        SELECT
            dt.time_id,
            dc.country_id,
            td.cov_ratio,
            td.value,
            td.value_scaled,
            %s,
            %s,
            NOW()
        FROM tmp_cleaned_demand td
        JOIN dim_time dt
            ON dt.datetime_utc = td.datetime_utc
        JOIN dim_country dc
            ON dc.country_name COLLATE utf8mb4_unicode_ci =
               td.country_code COLLATE utf8mb4_unicode_ci
        LEFT JOIN fact_demand fd
            ON fd.time_id = dt.time_id
           AND fd.country_id = dc.country_id
        WHERE fd.demand_id IS NULL;
    """, (batch_id, pipeline_id))

    fact_demand_inserted = cursor.rowcount
    conn.commit()

    log(f"fact_demand load complete | Inserted: {fact_demand_inserted} / {fact_demand_source}")
    log(f"fact_demand runtime: {time.time() - t0:.1f}s")

    # ---------------------------------------------------------
    # FACT_PRICE
    # ---------------------------------------------------------
    log("Counting fact_price rows to insert...")

    cursor.execute("""
        SELECT COUNT(*)
        FROM tmp_cleaned_price tp
        JOIN dim_time dt ON dt.datetime_utc = tp.datetime_utc
        JOIN dim_country dc
            ON dc.country_name COLLATE utf8mb4_unicode_ci =
               tp.country_name COLLATE utf8mb4_unicode_ci
        LEFT JOIN fact_price fp
            ON fp.time_id = dt.time_id
           AND fp.country_id = dc.country_id
        WHERE fp.price_id IS NULL;
    """)
    fact_price_source = cursor.fetchone()[0]
    log(f"fact_price rows to insert: {fact_price_source}")

    t0 = time.time()
    log("Loading fact_price...")

    cursor.execute("""
        INSERT INTO fact_price (
            time_id,
            country_id,
            price_eur_mwhe,
            batch_id,
            pipeline_id,
            load_ts
        )
        SELECT
            dt.time_id,
            dc.country_id,
            tp.price_eur_mwhe,
            %s,
            %s,
            NOW()
        FROM tmp_cleaned_price tp
        JOIN dim_time dt
            ON dt.datetime_utc = tp.datetime_utc
        JOIN dim_country dc
            ON dc.country_name COLLATE utf8mb4_unicode_ci =
               tp.country_name COLLATE utf8mb4_unicode_ci
        LEFT JOIN fact_price fp
            ON fp.time_id = dt.time_id
           AND fp.country_id = dc.country_id
        WHERE fp.price_id IS NULL;
    """, (batch_id, pipeline_id))

    fact_price_inserted = cursor.rowcount
    conn.commit()

    log(f"fact_price load complete | Inserted: {fact_price_inserted} / {fact_price_source}")
    log(f"fact_price runtime: {time.time() - t0:.1f}s")

    # ---------------------------------------------------------
    # SUMMARY
    # ---------------------------------------------------------
    log("---------------------------------------------------------")
    log("FACT LOAD SUMMARY")
    log(f"fact_demand inserted: {fact_demand_inserted}")
    log(f"fact_price inserted:  {fact_price_inserted}")
    log("---------------------------------------------------------")

    cursor.close()
    conn.close()
    log("Connection closed. FACT load complete.")

except Exception as e:
    log(f"ERROR: {e}. Rolling back transaction.")
    try:
        conn.rollback()
    except:
        pass
    try:
        cursor.close()
        conn.close()
    except:
        pass
    raise
